# Data & Recovery

**Perishable Demand Forecasting & Zero-Waste Inventory Engine** - FreshRetailNet-50K.

All logic lives in `src/`; this is a thin control surface.

| Section | What it does | src module |
|---|---|---|
| 1 | ingest -> schema proof -> the working subset | `data_io` |
| 2 | baselines on raw sales - the bar to clear | `baselines` |

The one fact everything rests on: **a sold-out shelf recording zero sales is not zero demand.**
The till under-counts exactly on the days a product sells best, so a model trained on raw sales
learns to keep ordering too little - and the shelf keeps emptying.

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, "..")   # notebooks/ is one level down, so the repo root has to go on the path

import pandas as pd

from src.utils import config, data_io
from src import baselines

## 1. Data - the working subset

`N_STORES` stores are drawn at random (seeded from `config.RANDOM_STATE`) and **every series they
carry is kept**, across all 32 categories. A series is one `(store_id, product_id)` pair - one
product in one store, followed daily.

Two reasons for sampling whole stores rather than individual series:

- **Representative.** Nothing is selected on sales volume, so the subset mirrors the corpus
  (~0.9 units/day, ~45% censored days) and needs no scope restriction declared when reporting.
- **Complete baskets.** Each store keeps its full assortment (~54 products), so the ordering stage can
  recommend a realistic order across a real shelf instead of two products.

`build_subset()` re-verifies the schema against the raw bytes, aggregates to daily, and persists
the daily subset + hourly companion + per-series censoring rates under `data/processed/`.

Changing `N_STORES` or the seed changes the subset, so **every downstream artifact must be
rebuilt after.**

In [ ]:
N_STORES     = 30     # stores drawn at random; ~51 series each
REFRESH_DATA = True  # rebuild from Hugging Face - OVERWRITES data/processed/

if REFRESH_DATA:
    data_io.build_subset(n_stores=N_STORES)

## 2. Baselines on raw (censored) sales - the bar to clear

Fitted on **training** days, scored on **non-stockout validation** days (the dataset's own
convention).

The shipped eval split is the **test week** and is not read here - it is opened once, at the final evaluation.
Both windows are cut from the train file by the dates in `config`.

Observe **WPE**: negative means systematic under-forecasting, which is exactly the censoring bias
the recovery layer removes.

In [ ]:
RUN_BASELINES = True 

if RUN_BASELINES:
    # Load the daily data and build the baseline scorecard
    daily = data_io.load("daily")
    # Build the baseline scorecard
    board = baselines.build_scorecard(daily)
else:
    board = pd.read_csv(config.BASELINE_SCORECARD, index_col=0)
board.round(4)